# 03.2 — Multimodal understanding lab

**Prerequisites**

1. `gpt-4o` deployed as `MODEL_CHAT` (from `00_setup`).
2. Optional sections need extra packages, both guarded by a flag:
   `pip install azure-ai-vision-imageanalysis` and
   `pip install --pre azure-ai-contentunderstanding`.

**Cost.** Well under $1. Images are billed as tokens, and `detail` is the dial —
section 2 measures exactly how much.

This lab uses `MODEL_CHAT` rather than `MODEL_MINI`. `gpt-4o-mini` does accept
images, but detail recall on charts and cluttered scenes is visibly weaker, and
several exercises here depend on it. Section 2 lets you compare them directly.

Working images go into `lab_output/`, which is git-ignored.

## 1. Setup and test assets

We use two public sample images so the notebook runs anywhere: a **chart** (a
complex image, for the accessibility section) and a **photograph** (for captioning
and object detection). Both are downloaded locally so we can exercise the base64
path, which is the one that matters for private data.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'scripts'))
from ai103 import cfg, credential, chat_client, project_client, show_usage

OUT = pathlib.Path.cwd() / 'lab_output'
OUT.mkdir(exist_ok=True)

VISION_MODEL = cfg.require('MODEL_CHAT')   # gpt-4o
MINI_MODEL = cfg.require('MODEL_MINI')
client = chat_client()
print('vision model:', VISION_MODEL)

In [ ]:
import requests

ASSETS = 'https://raw.githubusercontent.com/Azure-Samples/azure-ai-content-understanding-assets/main/'
CHART_URL = ASSETS + 'image/pieChart.jpg'
VIDEO_URL = ASSETS + 'videos/sdk_samples/FlightSimulator.mp4'
PHOTO_URL = 'https://raw.githubusercontent.com/Azure-Samples/cognitive-services-sample-data-files/master/ComputerVision/Images/objects.jpg'


def fetch(url, name):
    path = OUT / name
    if not path.exists():
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        path.write_bytes(r.content)
    print(f'{name:<14} {path.stat().st_size / 1024:6.0f} KB')
    return path


chart = fetch(CHART_URL, 'chart.jpg')
photo = fetch(PHOTO_URL, 'photo.jpg')

### Getting an image into a chat request

Two forms, and the choice is about **reachability**, not preference:

| | Public URL | Base64 data URI |
|---|---|---|
| Azure must be able to fetch it | yes | no |
| Works for private Blob Storage | no | yes |
| Request payload | tiny | ~33% larger than the file |

A private blob URL will **not** work — the service fetches the URL server-side with
no credentials of yours. Either mint a short-lived SAS, or read the bytes with your
managed identity and inline them. The helper below does the latter.

In [ ]:
import base64, mimetypes, json


def image_part(path, detail='auto'):
    """A chat content part carrying a local image as a base64 data URI."""
    mime = mimetypes.guess_type(str(path))[0] or 'image/jpeg'
    b64 = base64.b64encode(pathlib.Path(path).read_bytes()).decode()
    return {'type': 'image_url',
            'image_url': {'url': f'data:{mime};base64,{b64}', 'detail': detail}}


def url_part(url, detail='auto'):
    """A chat content part referencing a publicly reachable image."""
    return {'type': 'image_url', 'image_url': {'url': url, 'detail': detail}}


def look(parts, system=None, model=None, **kw):
    msgs = ([{'role': 'system', 'content': system}] if system else [])
    msgs.append({'role': 'user', 'content': parts})
    r = client.chat.completions.create(model=model or VISION_MODEL, messages=msgs, **kw)
    return r

## 2. `detail` is the cost dial

`low` sends one small pass over the image at a fixed, small token cost. `high`
additionally tiles the image into crops and bills every tile. Same question, same
image, two very different bills — and often the same answer.

In [ ]:
QUESTION = 'What kind of chart is this, and what is the largest segment?'

for detail in ('low', 'high'):
    r = look([{'type': 'text', 'text': QUESTION}, image_part(chart, detail)], temperature=0)
    print(f"--- detail={detail} ---")
    print(r.choices[0].message.content.strip()[:300])
    show_usage(r)
    print()

Compare the `prompt` token counts. The gap is the whole story.

| Use | `detail` |
|---|---|
| "Is there a person in this frame?" — classification, gist | `low` |
| Sampling one frame per second of video | `low`, always |
| Reading small text, counting objects, chart values | `high` |
| Not sure, low volume | `auto` |

> **Exam note.** In a multi-turn conversation the image stays in the message
> history and is **re-billed on every subsequent turn**. Long vision chats get
> expensive silently. Drop old images from the history once you have extracted what
> you need from them.

### Does the cheaper model do?

Run the same question through `gpt-4o-mini`. On a simple photo it is usually fine;
on charts and cluttered scenes it drops detail. Measure, do not assume.

In [ ]:
for model in (MINI_MODEL, VISION_MODEL):
    r = look([{'type': 'text', 'text': 'List every distinct percentage label shown on this chart.'},
              image_part(chart, 'high')], model=model, temperature=0)
    print(f'--- {model} ---')
    print(r.choices[0].message.content.strip()[:300])
    print()

## 3. Captions — concise, detailed, and across a set

The only thing separating a concise caption from a detailed one is how hard you
constrain the output. Models default to verbose.

In [ ]:
CONCISE = ('Write one caption of at most 15 words. No preamble, no trailing period '
           'commentary, no phrases such as "an image of". Output the caption only.')

DETAILED = ('Describe the image under exactly these headings, one short paragraph each:\n'
            'Subject / Setting / Composition / Notable detail.\n'
            'State only what is visible.')

for label, system in (('CONCISE', CONCISE), ('DETAILED', DETAILED)):
    r = look([{'type': 'text', 'text': 'Caption this image.'}, image_part(photo, 'low')],
             system=system, temperature=0.2)
    print(f'--- {label} ---')
    print(r.choices[0].message.content.strip())
    print()

### Several images in one call

Put every image in the **same** user message. Sent as separate calls, the model
cannot compare them — and comparison is the only reason to batch.

Number them in the text so the model can refer to them unambiguously. Note that
cost scales with the number of images, so `detail='low'` earns its keep here.

In [ ]:
parts = [
    {'type': 'text', 'text': 'Image 1:'}, image_part(chart, 'low'),
    {'type': 'text', 'text': 'Image 2:'}, image_part(photo, 'low'),
    {'type': 'text', 'text': 'For each image give a caption of at most 12 words, '
                             'labelled "1:" and "2:". Then one sentence on how they differ '
                             'in purpose. Output nothing else.'},
]
r = look(parts, temperature=0)
print(r.choices[0].message.content.strip())
show_usage(r)

## 4. Visual question answering, grounded in the evidence

Unconstrained, a multimodal model answers plausible questions from world knowledge
and presents the answer as observation. Grounding is three mechanisms working
together:

1. An explicit **refusal path** in the system prompt.
2. A **structured answer** that carries its own evidence, so the claim is auditable.
3. Low temperature, so it does not embellish.

The last question below is deliberately unanswerable from the pixels. A grounded
system must say so.

In [ ]:
VQA_SYSTEM = (
    'You answer questions about an image using ONLY what is visible in it. '
    'Never use outside knowledge or inference about things not shown. '
    'If the image does not contain enough evidence, set answer to "NOT_VISIBLE". '
    'Reply with JSON only: '
    '{"answer": string, "evidence": string, "confidence": "high"|"medium"|"low"}'
)

QUESTIONS = [
    'What type of chart is shown?',
    'What is the smallest segment and its percentage?',
    'Which company produced the data in this chart?',   # not in the image
]

for q in QUESTIONS:
    r = look([{'type': 'text', 'text': q}, image_part(chart, 'high')],
             system=VQA_SYSTEM, temperature=0,
             response_format={'type': 'json_object'})
    ans = json.loads(r.choices[0].message.content)
    print(f'Q: {q}')
    print(f"   answer     : {ans['answer']}")
    print(f"   evidence   : {ans['evidence']}")
    print(f"   confidence : {ans['confidence']}\n")

The third question should return `NOT_VISIBLE`. If it does not, the refusal path
is not strong enough — that is a prompt bug, and it is exactly the failure mode a
grounding evaluator (unit 02.1) is designed to catch.

### A verification pass

For high-stakes answers, re-ask with only the claim and the image. A second,
narrower judgement catches confident fabrication that a single pass will not.

In [ ]:
def verify(claim, image_path):
    r = look(
        [{'type': 'text', 'text': f'Statement: "{claim}"\nIs this statement directly '
                                  'supported by the image? Answer YES or NO and nothing else.'},
         image_part(image_path, 'high')],
        temperature=0,
    )
    return r.choices[0].message.content.strip().upper().startswith('YES')


for claim in ('The chart is a pie chart.', 'The chart shows quarterly revenue in dollars.'):
    print(f'{verify(claim, chart)!s:<6} {claim}')

## 5. Accessibility — alt text and extended descriptions

WCAG 2.2 SC **1.1.1 Non-text Content** asks for a text alternative serving the
*equivalent purpose*, and the right answer depends on the image's role:

| Role | Needs |
|---|---|
| Decorative | `alt=""` — present but empty, so it is skipped |
| Informative | Short alt, conventionally ≤ 125 characters |
| Functional (link/button) | Describe the **action**, not the picture |
| Complex (chart, diagram, map) | Short alt naming what it is, **plus** an extended description nearby |
| Text in image | The text itself |

The model needs to be told the rules and, crucially, **the page context** — the same
photograph needs different alt text on a news article and in a camera review. That
context dependence is why alt-text generation is an assistive tool with a human in
the loop, not a compliance checkbox.

In [ ]:
ALT_SYSTEM = """You write accessibility text following WCAG 2.2 SC 1.1.1.

Rules:
- Never begin with "Image of", "Picture of", "Photo of" or similar.
- Describe, do not evaluate. No aesthetic judgements.
- alt_text: at most 125 characters, plain text, no markup.
- role: one of decorative | informative | functional | complex.
- If role is decorative, alt_text must be the empty string.
- If role is complex, also write extended_description: for a chart give chart type,
  axes or categories with units, the overall trend, and the notable values. Do not
  list every data point unless the page is about the data.
- Otherwise extended_description is null.
- Use the page context to decide what matters.

Reply with JSON only:
{"role": string, "alt_text": string, "extended_description": string|null}"""


def alt_text_for(path, context, detail='high'):
    r = look([{'type': 'text', 'text': f'Page context: {context}'}, image_part(path, detail)],
             system=ALT_SYSTEM, temperature=0,
             response_format={'type': 'json_object'})
    return json.loads(r.choices[0].message.content)


result = alt_text_for(chart, 'An HR report page analysing how many hours per week staff work.')
print('role  :', result['role'])
print(f"alt   : {result['alt_text']}  ({len(result['alt_text'])} chars)")
print('long  :', (result['extended_description'] or '(none)')[:600])

### Context changes the answer

Same image, two different pages. If the alt text does not change, your prompt is
not actually using the context — and context-blind alt text is the most common way
automated accessibility fails an audit while passing a linter.

In [ ]:
CONTEXTS = [
    'A blog post about the specific distribution of weekly working hours.',
    'A decorative banner at the top of a page about office furniture. The image carries no information.',
]
for ctx in CONTEXTS:
    out = alt_text_for(chart, ctx, detail='low')
    print(f'context : {ctx[:60]}...')
    print(f"role    : {out['role']}")
    print(f"alt     : {out['alt_text']!r}\n")

A cheap deterministic check to run in CI. Notice it validates conventions a model
cannot be trusted to hold: an LLM will drift back to "This image shows…" the moment
your prompt is edited.

In [ ]:
BANNED = ('image of', 'picture of', 'photo of', 'this image', 'graphic of', 'icon of')


def lint_alt(entry):
    problems = []
    alt, role = entry['alt_text'], entry['role']
    if role == 'decorative' and alt != '':
        problems.append('decorative images must have empty alt text')
    if role != 'decorative':
        if not alt:
            problems.append('non-decorative image has no alt text')
        if len(alt) > 125:
            problems.append(f'alt text is {len(alt)} chars, convention is <= 125')
        if any(alt.lower().startswith(b) for b in BANNED):
            problems.append('alt text starts with a redundant phrase')
    if role == 'complex' and not entry.get('extended_description'):
        problems.append('complex image needs an extended description')
    return problems or ['ok']


print(lint_alt(result))

## 6. Objects and regions — why you need Azure AI Vision

First, watch the LLM fail at the one thing it cannot do. Ask `gpt-4o` for bounding
boxes and it will produce numbers. They will look plausible. They will not be
reliable, and there is no confidence score to tell you which ones to distrust.

In [ ]:
r = look([{'type': 'text', 'text': 'List every distinct object and give a pixel bounding box '
                                   '[x, y, w, h] for each. JSON array only.'},
          image_part(photo, 'high')], temperature=0)
print(r.choices[0].message.content.strip()[:600])
print('\nPlausible-looking. Unverifiable. No confidence scores. Do not ship this.')

Image Analysis 4.0 returns real coordinates with confidence. `pip install
azure-ai-vision-imageanalysis` to run the next cell.

| Visual feature | Returns |
|---|---|
| `OBJECTS` | Boxes + tag + confidence |
| `PEOPLE` | Boxes for people |
| `TAGS` | Tags + confidence, no coordinates |
| `CAPTION` / `DENSE_CAPTIONS` | One caption / per-region captions **with boxes** |
| `READ` | OCR with line and word polygons |
| `SMART_CROPS` | Crop rectangles preserving the region of interest |

`CAPTION` and `DENSE_CAPTIONS` are only available in a subset of regions.

In [ ]:
RUN_VISION = False  # pip install azure-ai-vision-imageanalysis

objects_found = []
if RUN_VISION:
    from azure.ai.vision.imageanalysis import ImageAnalysisClient
    from azure.ai.vision.imageanalysis.models import VisualFeatures

    # Image Analysis lives on the Foundry (AIServices) endpoint and accepts Entra ID.
    vision = ImageAnalysisClient(
        endpoint=cfg.require('AZURE_OPENAI_ENDPOINT').replace('.openai.azure.com',
                                                              '.cognitiveservices.azure.com'),
        credential=credential(),
    )
    analysis = vision.analyze(
        image_data=photo.read_bytes(),
        visual_features=[VisualFeatures.CAPTION, VisualFeatures.DENSE_CAPTIONS,
                         VisualFeatures.OBJECTS, VisualFeatures.TAGS],
    )
    if analysis.caption:
        print(f'caption: {analysis.caption.text}  (confidence {analysis.caption.confidence:.2f})')
    for o in (analysis.objects.list if analysis.objects else []):
        tag = o.tags[0]
        objects_found.append((tag.name, tag.confidence, o.bounding_box))
        print(f'  {tag.name:<16} {tag.confidence:.2f}  {o.bounding_box}')
else:
    print('skipped - set RUN_VISION = True after installing azure-ai-vision-imageanalysis')

### The pattern that actually ships

Combine them. **Image Analysis for coordinates, the LLM for meaning.** Pass the
detections in as text alongside the image and ask the model to reason over them:
you get boxes you can render in a UI and prose a human wants to read, and the
model's reasoning is anchored to detections that carry confidence scores.

In [ ]:
if objects_found:
    detections = '\n'.join(f'- {n} (confidence {c:.2f}) at {b}' for n, c, b in objects_found)
    r = look([{'type': 'text',
               'text': 'A detector found these objects:\n' + detections +
                       '\n\nUsing the image and this list, describe the scene in two sentences '
                       'and flag any detection you think is wrong.'},
              image_part(photo, 'high')], temperature=0)
    print(r.choices[0].message.content.strip())
else:
    print('run the previous cell with RUN_VISION = True first')

## 7. Content Understanding

Content Understanding turns media into **your** schema, with confidence scores and
grounding — the two things a raw LLM prompt does not give you, and the two things
straight-through processing needs.

Its API is asynchronous, like everything that processes media:

```
POST {endpoint}/contentunderstanding/analyzers/{id}:analyze?api-version=2025-11-01
  -> 202 + Operation-Location header
GET  {Operation-Location}  -> poll until status = Succeeded
```

The endpoint is the **Foundry (AIServices) resource** endpoint, not the project
endpoint, and the resource needs default model deployments configured once (Content
Understanding Studio does this for you). `2025-11-01` is GA; `2026-06-01-preview`
carries the preview features.

The REST version below has no SDK dependency.

In [ ]:
import time

RUN_CU = False  # requires a Foundry resource in a Content Understanding region
CU_API = '2025-11-01'
CU_ENDPOINT = (cfg.get('AZURE_CONTENT_UNDERSTANDING_ENDPOINT')
               or cfg.get('AZURE_AI_INFERENCE_ENDPOINT')
               or cfg.get('AZURE_OPENAI_ENDPOINT', '')).rstrip('/')


def cu_analyze(analyzer_id, url, poll_every=3, timeout=600):
    """Submit an analyze request, poll Operation-Location, return the result JSON."""
    tok = credential().get_token('https://cognitiveservices.azure.com/.default').token
    hdrs = {'Authorization': f'Bearer {tok}', 'Content-Type': 'application/json'}
    submit = f'{CU_ENDPOINT}/contentunderstanding/analyzers/{analyzer_id}:analyze?api-version={CU_API}'

    r = requests.post(submit, headers=hdrs, json={'url': url}, timeout=60)
    r.raise_for_status()
    op = r.headers['Operation-Location']

    deadline = time.time() + timeout
    while True:
        if time.time() > deadline:
            raise TimeoutError(f'{analyzer_id} did not finish in {timeout}s')
        body = requests.get(op, headers={'Authorization': f'Bearer {tok}'}, timeout=60).json()
        if body.get('status') in ('Succeeded', 'Failed'):
            return body
        time.sleep(poll_every)


if RUN_CU:
    out = cu_analyze('prebuilt-imageSearch', CHART_URL)
    content = out['result']['contents'][0]
    print('markdown:', content.get('markdown', '')[:300])
    for name, field in (content.get('fields') or {}).items():
        print(f"  {name}: {str(field.get('valueString') or field.get('value'))[:200]}")
else:
    print('skipped - set RUN_CU = True')

### The same thing with the SDK

`pip install --pre azure-ai-contentunderstanding` (preview). The SDK hides the
polling behind a long-running-operation poller, which is the pattern every Azure
SDK uses for async operations.

In [ ]:
RUN_CU_SDK = False

if RUN_CU_SDK:
    from azure.ai.contentunderstanding import ContentUnderstandingClient
    from azure.ai.contentunderstanding.models import AnalysisInput

    cu = ContentUnderstandingClient(endpoint=CU_ENDPOINT, credential=credential())

    poller = cu.begin_analyze(analyzer_id='prebuilt-imageSearch',
                              inputs=[AnalysisInput(url=CHART_URL)])
    result = poller.result()
    content = result.contents[0]
    print(content.markdown[:300])
    summary = content.fields.get('Summary')
    if summary is not None:
        print('\nSummary:', getattr(summary, 'value', summary))
else:
    print('skipped - set RUN_CU_SDK = True')

### Standard vs pro mode

| | Standard (default) | Pro |
|---|---|---|
| Inputs per request | one | **many**, reasoned over together |
| Reasoning | single pass | multi-step |
| External knowledge base | no | yes — link, enrich, validate |
| Cost / latency | lower | higher |

The decision rule is about **inputs**, not schema size: if answering the question
requires comparing one asset against another asset or against reference data, that
is pro mode. An inspection photograph checked against the equipment specification
sheet is pro. A thousand independent product photos, each yielding the same eight
fields, is standard — and far cheaper.

In the SDK, pro mode is expressed by passing several `AnalysisInput` objects to a
pro-mode analyzer:

```python
poller = cu.begin_analyze(
    analyzer_id='my-pro-analyzer',           # created with mode='pro'
    inputs=[AnalysisInput(url=photo_url), AnalysisInput(url=spec_sheet_url)],
)
```

Pro mode arrived in the `2025-05-01-preview` API. Treat it as **preview**.

## 8. Video: segments, not frames

Two approaches, and the exam wants you to justify the choice.

### 8a. Sample frames and send them to the LLM

Right for short clips and one-off questions. Wrong at scale: it costs tokens per
frame, throws away the audio, and cannot tell you where one scene ends and the next
begins.

Note `detail='low'` — with N frames per request, this is not a micro-optimisation.
The cell needs `ffmpeg` on your PATH; skip it if you do not have it.

In [ ]:
import shutil, subprocess

RUN_FRAMES = shutil.which('ffmpeg') is not None

if RUN_FRAMES:
    video = fetch(VIDEO_URL, 'clip.mp4')
    frames_dir = OUT / 'frames'
    frames_dir.mkdir(exist_ok=True)
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(video),
                    '-vf', 'fps=1/3,scale=512:-1', str(frames_dir / 'f_%03d.jpg')], check=True)
    frames = sorted(frames_dir.glob('*.jpg'))[:6]
    print(f'{len(frames)} frames sampled')

    parts = []
    for i, f in enumerate(frames, 1):
        parts += [{'type': 'text', 'text': f'Frame {i} (t={i * 3 - 3}s):'}, image_part(f, 'low')]
    parts.append({'type': 'text', 'text': 'These are ordered frames from one clip. Summarise what '
                                          'happens in three sentences, and name the frame where the '
                                          'scene changes most.'})
    r = look(parts, temperature=0)
    print(r.choices[0].message.content.strip())
    show_usage(r)
else:
    print('ffmpeg not found - skipping frame sampling')

### 8b. Content Understanding video analysis

`prebuilt-videoSearch` returns **one content object per detected segment**, each
with `startTimeMs` / `endTimeMs`, Markdown, transcript and your custom fields. That
structure is what makes video searchable: index each segment and a retrieval hit
points at a timestamp rather than at a two-hour file. Unit 05.1 uses exactly this
as an ingestion source.

In [ ]:
if RUN_CU:
    out = cu_analyze('prebuilt-videoSearch', VIDEO_URL, timeout=900)
    for seg in out['result']['contents']:
        start, end = seg.get('startTimeMs', 0), seg.get('endTimeMs', 0)
        print(f"--- segment {start / 1000:.1f}s -> {end / 1000:.1f}s "
              f"({seg.get('width')}x{seg.get('height')}) ---")
        print(seg.get('markdown', '')[:300], '\n')
else:
    print('skipped - set RUN_CU = True')

## 9. Choosing a service — the summary you should be able to reproduce

| Requirement | Winner | Because |
|---|---|---|
| "Explain what is happening and why it matters" | Multimodal LLM | Reasoning and free-form output |
| "Draw a box around every helmet" | Azure AI Vision `OBJECTS` | LLMs do not produce reliable coordinates |
| "Same 12 fields from 50,000 product photos, with confidence" | Content Understanding | Schema, confidence, grounding, batch |
| "Read the serial number off this label" | Vision `READ` (OCR), or Document Intelligence | Purpose-built, cheap, deterministic |
| "Cross-check the inspection photo against the spec sheet" | Content Understanding **pro mode** | Multiple inputs reasoned over together |
| "Transcript plus scene segmentation for a 2-hour video" | Content Understanding video | Native segments and transcript |
| "Answer arbitrary user questions about one uploaded image" | Multimodal LLM | Open-ended; no fixed schema |
| "Alt text for a CMS, reviewed by an editor" | Multimodal LLM | Needs page context and judgement |
| "Count people crossing a line in a live camera feed" | Vision Spatial Analysis | Real-time, geometric |

> **Exam note.** Watch for scenarios that sound generative and end with a
> requirement that is not: "describe the shelf" is an LLM job, but "and highlight
> each product" needs coordinates. The correct answer is usually **both services**,
> not the more impressive one.

## Exercise

Solutions are in [quiz.md](quiz.md).

1. **Measure `detail`.** Run the same question over the same image at `low`, `high`
   and `auto`, five times each at `temperature=0`. Report mean prompt tokens per
   setting and whether the answers agreed. Then state the rule you would give a
   team for choosing between them.

2. **A grounded VQA function.** Write `ask_image(question, image_path) -> dict`
   that returns `{answer, evidence, confidence, verified}` where `verified` is the
   result of a second verification pass, and which returns `NOT_VISIBLE` rather
   than guessing. Prove it refuses on a question the image cannot answer.

3. **An accessibility pipeline.** Given a list of `(image_path, page_context)`
   pairs, produce a report with role, alt text, extended description where
   required, and every lint failure from `lint_alt`. Make one image decorative and
   confirm it gets empty alt text.

4. **Argue a design.** A retailer wants to process 200,000 product photos into
   twelve structured attributes, and separately let merchandisers ask ad-hoc
   questions about any single photo. Write the two-paragraph design and justify the
   service choice for each half, including what changes if they also need each
   attribute located in the image.

Use the empty cell below.

## Clean up

This lab creates nothing billable — no deployments, no analyzers, no hourly
resources. If you created a **custom analyzer** in Content Understanding Studio,
delete it there; analyzers are stored on the resource and are free to keep but easy
to lose track of.

Downloaded media and sampled frames sit in `lab_output/`, which is git-ignored.
Delete the folder if you want the disk space back.

In [ ]:
total = sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file())
print(f'{OUT} holds {total / 1e6:.1f} MB')
print('Nothing billable was created by this lab.')
print('If you built a custom analyzer in Content Understanding Studio, delete it there.')